# Homework 1
### Tasks: 
  1. Exercise 2.1
  2. Exercise 2.2
  3. Exercise 2.3
  4. Exercise 2.4

# Exercise 2.1
    "Implement the GGH/HNF cryptosystem. To do so,
    consider the generation of a “good” basis as described in the
    lecture notes." 
**Provided in the example (Lattice.ipynb) is a GGH/HNF Key Generation function that generates a random basis and calculates the Hadamard ratio.** `\n`

In the example provided a basic implementation of the GGH/HNF Cryptosystem is provided, however this can be expanded upon as per the lecture notes. The provided implementation has a random integer matrix $B$ ,  this has the advantages high entropy/randomness. However we can implement a structured method where $B = kI + R$ . Practically this has several advantages in predictable behavior with bounded entries, deterministic time and faster computation. This structured method can be seen in the lecture notes section 2.3.1 page 24-25. 

$H$ == PUBLIC KEY
$B$ == PRIVATE KEY



## Observations
Anecdotally with a structured basis, no matter how small our n value we can encrypt and decrypt successfully, however with the random basis when n is too small we get a failure to decrypt. With a structured basis I can reasonably generate keys with a matrix dimension size of up to 1500x1500, however with random basis anything larger than 100 can take hours to generate in some cases due to the lack of deterministic structure. 

In [1]:
"""
Define lattice reduction, encryption, decryption and save/load key functions.
"""

def lattice_reduction(message,H_pub_key):
    n_matrix_dimension = len(message)
    for ii in range(n_matrix_dimension):
        message = message - (message[ii]//H_pub_key[ii][ii])*H_pub_key[ii]
    return message

def GGH_HNF_encryption(message,H_pub_key):
    return lattice_reduction(message,H_pub_key)

def GGH_HNF_decryption(cyphertext,B_priv_key):
    aux = cyphertext*B_priv_key^(-1)
    return cyphertext - vector([ele.round() for ele in aux])*B_priv_key


"""
Here we have expanded upon the example code to provide the ability to generate keys keys with a structured basis,
and for the random basis I have added a parrallel version to leverage multiple CPU Threads. 
"""
def GGH_HNF_key_generation(n,l, lattice_type):
    match lattice_type:
        case "Structured":
            # Compute k = ⌈√n⌉ × l
            k = ceil(sqrt(n)) * l
            # Generate random matrix R with entries in {-l, ..., l}
            R = matrix(ZZ, n, n, lambda i, j: ZZ.random_element(-l, l+1))
            # Create structured basis
            B = k * identity_matrix(ZZ, n) + R

        case "Random":
            B = matrix(n,[(ZZ^n).random_element() for ii in range(n)])
            count = 0
            while RR(abs(det(B))/prod(B[ii].norm() for ii in range(n)))^(1/4) < 0.8:
                if count % 100 == 0:
                    ratio = RR(abs(det(B))/prod(B[ii].norm() for ii in range(n)))^(1/4)
                print(f"\rTry {count}: ratio = {ratio:.4f}", end='', flush=True)
                B = matrix(n,[(ZZ^n).random_element() for ii in range(n)])
                count += 1
            print(f"Success after {count} tries!")
            #H = B.hermite_form()
            #return (H,B)

        case "Random-Parrallel":
            @parallel(ncpus=Integer(12))  # Use 8 cores
            def generate_matrix(seed):
                seed = int(seed)  # Ensure seed is an integer
                set_random_seed(seed)
                B_candidate = matrix(n, [(ZZ^n).random_element() for ii in range(n)])
                ratio = RR(abs(det(B_candidate))/prod(B_candidate[ii].norm() for ii in range(n)))^(1/n)
                return (B_candidate, ratio) if ratio >= 0.8 else None
            
            B = None
            count = 0
            batch_size = 100
            
            while B is None:
                seeds = list(range(count, count + batch_size))  # Convert to list
                results = list(generate_matrix(seeds))
                
                for inp, output in results:
                    if output is not None:
                        B, ratio = output
                        print(f"\n✓ Success with ratio = {ratio:.4f} after ~{count} tries!")
                        break
                    
                count += batch_size
                print(f"\rTries: ~{count}", end='', flush=True)
    # Calculate Hadamard ratio
    det_B = abs(B.determinant())
    prod_norms = prod(RR(B[i].norm()) for i in range(n))
    hadamard_ratio = RR(det_B / prod_norms)^(1/n)

    # Compute HNF
    H = B.hermite_form()

    return (H, B, hadamard_ratio)


# Save keys
def save_keys(H, B, filename_prefix):
    # Save public key (for encryption)
    with open(f"{filename_prefix}_public.txt", 'w') as f:
        f.write(str(H))
    
    # Save private key (for decryption) - keep secure!
    with open(f"{filename_prefix}_private.txt", 'w') as f:
        f.write(str(B))

# Load keys
def load_keys(path):
    print(f"Loading keys from {path}")
    
    with open(f"{path}_public.txt", 'r') as f:
        H = matrix(ZZ, [[int(x) for x in line.strip().strip('[]').split()] for line in f if line.strip()])
    
    with open(f"{path}_private.txt", 'r') as f:
        B = matrix(ZZ, [[int(x) for x in line.strip().strip('[]').split()] for line in f if line.strip()])
    
    return H, B

def gen_test_save_keys(n, l, lattice_type):
    (H, B, hadamard_ratio) = GGH_HNF_key_generation(n, l, lattice_type)
    r = vector(ZZ, [ZZ.random_element(-5, 5) for _ in range(n)])
    c = GGH_HNF_encryption(r, H)
    r_recovered = GGH_HNF_decryption(c, B)
    if r == r_recovered:
        save_keys(H, B, f"./notebooks/keys/_{n}_{l}_{lattice_type}")
        works = "✓"
    else:
        works = "✗"
    return (H, B, hadamard_ratio, works)
    

    

In [2]:
"""
Lets run a one off test to make sure everything is working, 
without the all in one function to verify the outcomes of each individual step. 
"""
n = 50 # Dimension of the lattice -> n x n
l = 4 # Number of perturbations in the lattice (Only applies to the structured method)
lattice_type = "Structured" # Type of lattice to be used
import os

(H, B, hadamard_ratio) = GGH_HNF_key_generation(n,l,lattice_type)
print(f"Hadamard ratio: {hadamard_ratio:.4f}")
print("✓ Lattice reduction, encryption and decryption functions defined")
print("✓ Running end-to-end encryption/decryption test")
r = vector(ZZ, [ZZ.random_element(-4, 4) for _ in range(n)])
c = GGH_HNF_encryption(r, H)
r_recovered = GGH_HNF_decryption(c, B)
print(f"✓ Works: {r == r_recovered}")
save_keys(H, B, "./notebooks/keys/")



Hadamard ratio: 0.8578
✓ Lattice reduction, encryption and decryption functions defined
✓ Running end-to-end encryption/decryption test
✓ Works: True


In [24]:
"""
Here I am defining a test set to generate keys, and create a table with pandas dataframe for ease of observability.
We use the gen_test_save_keys function to generate keys, test encrypting and decrypting with them then save them to a folder for future use. 
"""
from dataclasses import dataclass
import pandas as pd
from IPython.display import display, clear_output
import time

@dataclass
class TestSet:
    size: str
    value: int

# Usage
nset: list[TestSet] = [
    TestSet("xx-small", 82),
    #TestSet("x-small", 78),
    #TestSet("small", 79), 
    #TestSet("medium", 80),
    #TestSet("xx-small", 77),
    #TestSet("x-small", 78),
    #TestSet("small", 79), 
    #TestSet("medium", 80),
    #TestSet("huge", 5000),
]

# Initialize DataFrame once
df = pd.DataFrame(columns=['n Value', 'l', 'hadamard_ratio', 'works?'])

def test_harness(nset, l=8, lattice_type="structured"):
    global df
    
    for test_case in nset:  # Changed from TestSet to test_case
        start = time.time()
        (H, B, hadamard_ratio, works) = gen_test_save_keys(test_case.value, l=l, lattice_type=lattice_type)
        
        new_row = pd.DataFrame([{  # Note: wrap dict in list for single row
            'n Value': test_case.size,
            'l': l,
            'hadamard_ratio': hadamard_ratio,
            'works?': works,
        }])
        
        df = pd.concat([df, new_row], ignore_index=True)
        
        # Display updated table
        clear_output(wait=True)
        display(df)

test_harness(nset=nset, l=8, lattice_type="Structured")

,n Value,l,hadamard_ratio,works?
0,xx-small,8,0.873875793210128,✓


# Exercise 2.2
    "Perform an analysis of how LLL can be used to
    attack GGH/HNF cryptosystem. I suggest to make different tries
    for different parameters and elections of keys."
LLL Attack relies on having a private basis $B$ having a high hadamard ratio, if this is the case the attack can recover the private key $B$ from the public key $H$. 

The hadamard ratio of a matrix $B$ is defined as $\frac{|det(B)|}{|B|}$. 

In the case of the lattice basis $B$ the hadamard ratio is $\frac{|det(B)|}{|B|} = \frac{|det(B)|}{|B|^2}$. 

The lattice basis $B$ is a random matrix with entries in $\mathbb{Z}$ and $\mathbb{Q}$. 

We can use LLL to find a basis $B$ with a high hadamard ratio.

## structured lattices
All lattices in the current tests have:
* a structured basis $B = kI + R$, where $k$ is a constant and $R$ is a random matrix with entries in $\mathbb{Z}$ and $\mathbb{Q}$.
* a Hadamard ratio threshold of $< 0.8$ (Quite high)
* an l value of 4 (l is the amount of random pertubations, an l of 4 means entries range from ${-4, ..., 4}$, theoretically a larger l value is more secure against LLL but is harder to decrypt.)

## Testing
In the previous example we were encrypting and decrypting a random matrix of numbers,
however for this example we will generate a matrix to use as our test data, and use the same matrix across an entire batch of tests to maintain integrity. with all fixed variables described previously and a fixed input message, the only variable is the matrix dimension which we are testing. 


In [9]:
"""
Define functions to generate test data and to perform the LLL attack. 
"""

def generate_test_input_matrix(n, l): #n is the dimension of the matrix n x n, l is the pertubation size so the range (-l, ..., l)
    message = vector(ZZ, [ZZ.random_element(-l, l) for _ in range(n)])
    with open(f"./notebooks/test_matrix.txt", 'w') as f:
        f.write(str(message))
    return message

def compute_hadamard_ratio(B, n):
    det_B = abs(B.determinant())
    prod_norms = prod(RR(B[i].norm()) for i in range(n))
    hadamard_ratio = RR(det_B / prod_norms)^(1/n)
    return hadamard_ratio

def LLL_attack(H_pub_key):
    B_reduced = H_pub_key.LLL(delta=0.9999, fp='rr', perc='1000', algorithm='fpLLL:proved')
    return B_reduced
  

In [10]:
m = generate_test_input_matrix(50, 4)

print(len(m))

50


In [11]:
import time
def LLL_single_attack(matrix_dimension, l_range):
    message = generate_test_input_matrix(matrix_dimension, l_range) #(size of matrix, pertubations)
    H_pub_key, B_priv_key = load_keys(f"./notebooks/keys/_{matrix_dimension}_{l_range}_Structured")
    encrypted_message = GGH_HNF_encryption(message, H_pub_key)
    decrypted_message = GGH_HNF_decryption(encrypted_message, B_priv_key)
    original_hadamard_ratio = compute_hadamard_ratio(B_priv_key, matrix_dimension)
    start_time = time.time()
    B_reduced = LLL_attack(H_pub_key)
    end_time = time.time()
    attack_duration = end_time - start_time
    cracked_hadamard_ratio = compute_hadamard_ratio(B_reduced, matrix_dimension)
    print(f"original_hadamard_ratio = {original_hadamard_ratio}")
    print(f"cracked_hadamard_ratio = {cracked_hadamard_ratio}")
    cracked_message = GGH_HNF_decryption(encrypted_message, B_reduced)
    result = ""
    if message == cracked_message:
        result = "✓ Success!"

    else:
        result = "✗ Failed!"
    
    print(result)
    return result, original_hadamard_ratio, cracked_hadamard_ratio, attack_duration, message

result, original_hadamard_ratio, cracked_hadamard_ratio, attack_duration, message  = LLL_single_attack(75, 4)



Loading keys from ./notebooks/keys/_75_4_Structured
original_hadamard_ratio = 0.854940998670458
cracked_hadamard_ratio = 0.854940998670458
✓ Success!


In [26]:
from dataclasses import dataclass
import pandas as pd
from IPython.display import display, clear_output

@dataclass
class Test: 
    matrix_dimension: int
    l_range: int
    original_hadamard: float = None
    cracked_hadamard: float = None
    result: str = None
    duration: float = None

    def add_original_hadamard(self, value: float):
        self.original_hadamard = value

    def add_cracked_hadamard(self, value: float):
        self.cracked_hadamard = value
    
    def add_result(self, value: str):
        self.result = value

    def add_duration(self, value: float):
        self.duration = value
        

TestSet: list[Test] = [
    Test(83, 4),
    Test(83, 6),
    Test(83, 8),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 4),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(79, 8),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),
    #Test(83, 6),

]

df = pd.DataFrame(columns=[
    'matrix_dimension', 
    'l_range', 
    'original_hadamard', 
    'cracked_hadamard', 
    'result',
    'duration'
    ])


def LLL_attack_harness(TestSet):
    global df

    for test_case in TestSet:
        print(f"test_case.l_range {test_case.l_range}")
        result, original_hadamard_ratio, cracked_hadamard_ratio, attack_duration, message = LLL_single_attack(test_case.matrix_dimension, test_case.l_range)
        test_case.add_original_hadamard(original_hadamard_ratio)
        test_case.add_cracked_hadamard(cracked_hadamard_ratio)
        test_case.add_result(result)
        test_case.add_duration(attack_duration)
        new_row = pd.DataFrame([{
                'matrix_dimension': test_case.matrix_dimension, 
                'l_range': test_case.l_range , 
                'original_hadamard': test_case.original_hadamard, 
                'cracked_hadamard': test_case.cracked_hadamard, 
                'result': test_case.result,
                'duration': test_case.duration,
        }])

        df = pd.concat([df, new_row], ignore_index=True)

        clear_output(wait=True)
        display(df)
        

        
LLL_attack_harness(TestSet=TestSet)





,matrix_dimension,l_range,original_hadamard,cracked_hadamard,result,duration
0,83,4,0.855141464836459,0.218225075796452,✗ Failed!,4.738071
1,83,6,0.873002041463964,0.873002041463964,✓ Success!,5.607524
2,83,8,0.875984495500957,0.875984495500957,✓ Success!,5.585948


# Exercise 2.3
"Try to attack NTRU with LLL. Same considerations
that in previous exercise." 




